# Compare the 79NG Melt Rate between Model and Satellite Measurements

Notebook by Markus Reinert (IOW, 2024–2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
import numpy as np
import xarray as xr
import cmocean.cm as cmo
import matplotlib.pyplot as plt
from pyproj import CRS, Transformer

from tools.visualization import cm

## Prepare the coordinate transformation

### Ellipsoidal coordinates

The model results and the figure are in lat–lon coordinates.

In [ ]:
crs_latlon = CRS.from_epsg(4326)
crs_latlon

### Cartesian coordinates

The satellite measurements are in Cartesian coordinates.
In this coordinate system, distances at 79NG are close to physical distances on Earth.

In [ ]:
crs_cartesian = CRS.from_epsg(3413)
crs_cartesian

### CRS transformation

This transform function is used to have all datasets in the same lat–lon coordinate system for plotting.

In [ ]:
transform_xy_to_latlon = Transformer.from_crs(crs_cartesian, crs_latlon).transform

## Load the model result

In [ ]:
ds = xr.open_dataset("output/out_mean_compressed.nc")

# Convert single time value from coordinate to attribute
assert ds.time.size == 1
time_value = ds.time.data[0]
ds = ds.squeeze(dim="time", drop=True)
ds.attrs["time"] = time_value

# Add (missing) geometry variables to the dataset
with xr.open_dataset("output/out_geometry_2d.nc") as geo:
    for var in geo.variables:
        if var in ds:
            print(var, "exists in dataset and geometry.")
            # Check that the data matches between dataset and geometry
            xr.testing.assert_equal(ds[var], geo[var])
        else:
            ds[var] = geo[var]

# Compute the mask of the floating ice tongue
ds["ice_mask"] = ds.glIceD > 0
ds.ice_mask.attrs = {"long_name": "ice mask"}

# Convert the melt rate to m/yr
ds["meltrate"] = ds.vm.where(ds.ice_mask) * 3600 * 24 * 365.2425
ds.meltrate.attrs = {"long_name": "subglacial melt rate", "units": "m yr$^{-1}$"}

# Check grid spacing
dlat = 1 / 240
dlon = 3 / 120
assert np.allclose(np.diff(ds.latc), dlat)
assert np.allclose(np.diff(ds.lonc), dlon)

# Use this aspect ratio in lat–lon plots for squared grid cells; since dx and dy
# are both approx. 500 m, this aspect ratio gives an approximate equal-area map.
grille_carree = dlon / dlat

ds

## Load the measurements

### Wilson et al. (2017)

Reference: Wilson, N., Straneo, F., and Heimbach, P. Satellite-derived submarine melt rates and mass balance (2011–2015) for Greenland's largest remaining ice tongues. The Cryosphere (2017). https://doi.org/10.5194/tc-11-2773-2017

In [ ]:
with xr.open_dataset("data/Nioghalvfjerdsbrae-128m_20161101.nc") as ds_Wilson:
    # Check the grid spacing (computations in the following assume constant grid spacing)
    dx_Wilson = 128
    assert (np.diff(ds_Wilson.x) == dx_Wilson).all() and (np.diff(ds_Wilson.y) == dx_Wilson).all(), "grid spacing is not uniform."

    # Check the coordinate system
    crs_Wilson = ds_Wilson.Band1.grid_mapping
    assert crs_Wilson.replace("_", " ") in crs_cartesian.name.lower(),\
        f"grid mapping of dataset is {crs_Wilson!r} but should be {crs_cartesian} ({crs_cartesian.name})"

    # Compute and add ellipsoidal coordinates
    lat, lon = transform_xy_to_latlon(*xr.broadcast(ds_Wilson.x, ds_Wilson.y))
    ds_Wilson.coords["lon"] = (["x", "y"], lon)
    ds_Wilson.coords["lat"] = (["x", "y"], lat)
    ds_Wilson.lon.attrs = {"long_name": crs_latlon.axis_info[0].name, "units": "°E", "CRS": str(crs_latlon)}
    ds_Wilson.lat.attrs = {"long_name": crs_latlon.axis_info[1].name, "units": "°N", "CRS": str(crs_latlon)}

    # Process the melt rate
    meltrate_Wilson = -ds_Wilson.Band1.assign_attrs(long_name="melt rate", units="m yr$^{-1}$").rename("meltrate")

meltrate_Wilson.plot(figsize=(8, 8)); plt.gca().set_aspect("equal")
meltrate_Wilson

### Millan et al. (2023)

References:
 - Millan, R., Jager, E., Mouginot, J. et al. Rapid disintegration and weakening of ice shelves in North Greenland. Nat Commun 14, 6914 (2023). https://doi.org/10.1038/s41467-023-42198-2
 - Millan, R., Jager, E., Mouginot, J. et al. Dataset supporting "Rapid Disintegration and Weakening of Ice Shelves in North Greenland".  Zenodo (2023). https://doi.org/10.5281/zenodo.8354794

In [ ]:
ds_Millan = xr.open_dataset("data/Millan_2023/basal_melt_79n_v2022Nov.nc")

# Check the grid spacing (computations in the following assume constant grid spacing)
dx_Millan = 150
assert (np.diff(ds_Millan.x) == dx_Millan).all() and (np.diff(ds_Millan.y) == dx_Millan).all(), "grid spacing is not uniform."

# Compute and add ellipsoidal coordinates
lat, lon = transform_xy_to_latlon(*xr.broadcast(ds_Millan.x, ds_Millan.y))
ds_Millan.coords["lon"] = (["x", "y"], lon)
ds_Millan.coords["lat"] = (["x", "y"], lat)
ds_Millan.lon.attrs = {"long_name": crs_latlon.axis_info[0].name, "units": "°E", "CRS": str(crs_latlon)}
ds_Millan.lat.attrs = {"long_name": crs_latlon.axis_info[1].name, "units": "°N", "CRS": str(crs_latlon)}

ds_Millan.load()

The ice tongue area changes from year to year:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10), dpi=300)
for i, year in enumerate(ds_Millan.year):
    c = cmo.amp(i / (ds_Millan.year.size - 1))
    ds_Millan.h.sel(year=year).notnull().plot.contour(levels=[0.5], linewidths=0.5, colors=[c])
    ax.plot([], [], color=c, label=f"{year:d}")
ax.legend(ncols=4, loc="lower right")
ax.set(title="Extent of the 79NG ice tongue in each year", aspect="equal");

Mean melt rate for each year:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ds_Millan.h.mean(["x", "y"]).plot(ax=ax, marker="o")
ax.set_xticks(np.arange(2000, 2030, 5))
ax.set_xticks(ds_Millan.year, minor=True)
ax.set_xlim(ds_Millan.year[0], ds_Millan.year[-1])
ax.set(ylim=(0, 8), ylabel="melt rate in m/yr", title="Annual mean melt rate (Millan et al. 2023)")
ax.grid(ls=":")

We compare the model melt rate with the melt rate by Millan et al. (2023) for the years 2011 to 2020 (the period used for the model boundary conditions).
Since the ice tongue area changes, we consider the union of the ice tongue areas of the ten years.
At each position, we take the temporally averaged melt rate, skipping years with no data at a given position.
Consequently, the melt rate is averaged over a different number of years (between 1 and 10) at different locations.

In [ ]:
meltrate_Millan = ds_Millan.h.sel(year=slice(2011, 2020)).mean("year")
meltrate_Millan = meltrate_Millan.assign_attrs(long_name="melt rate", units="m yr$^{-1}$").rename("meltrate")
meltrate_Millan.plot(figsize=(7, 8)); plt.gca().set_aspect("equal")
meltrate_Millan

An alternative approach (implemented below) would be to take a melt rate of zero if a given point is not part of the ice tongue in a given year.
Then, the time-averaged melt rate would be computed on 10 years everywhere.
However, this gives a non-smooth melt rate near the grounding line, which does not look realistic.
Differences between these two approaches exist mostly near the grounding line and the calving fronts, to a smaller extent also near the fjord walls.

### Wang et al. (2024)

Reference:
Wang, G., et al. Accelerated Basal Melt Rates of Ice Shelves in North Greenland From 2013 to 2022 Estimated With the High-Resolution ArcticDEM. _JGR: Oceans_ (2024). https://doi.org/10.1029/2024JC021509

In [ ]:
with xr.open_dataset("data/Wang_2024_79N_2013-2022_mean.nc") as ds_Wang:
    # Check the grid spacing (computations in the following assume constant grid spacing)
    dx_Wang = 150
    assert (np.diff(ds_Wang.x) == dx_Wang).all() and (np.diff(ds_Wang.y) == -dx_Wang).all(), "grid spacing is not uniform"

    # Check the coordinate system
    assert crs_cartesian.name in ds_Wang.spatial_ref.crs_wkt, f"coordinate system is different from {crs_cartesian}"

    # Compute and add ellipsoidal coordinates
    lat, lon = transform_xy_to_latlon(*xr.broadcast(ds_Wang.x, ds_Wang.y))
    ds_Wang.coords["lon"] = (["x", "y"], lon, {"long_name": crs_latlon.axis_info[0].name, "units": "°E", "CRS": str(crs_latlon)})
    ds_Wang.coords["lat"] = (["x", "y"], lat, {"long_name": crs_latlon.axis_info[1].name, "units": "°N", "CRS": str(crs_latlon)})

    # Process the melt rate
    meltrate_Wang = ds_Wang.melt_rate.load()
    assert meltrate_Wang.units == "meters per year", "melt rate units are not as expected"
    assert "description" not in meltrate_Wang.attrs, "attribute description exists and would be overwritten"
    meltrate_Wang.attrs.update({"description": meltrate_Wang.long_name, "long_name": "melt rate", "units": "m yr$^{-1}$"})

meltrate_Wang.plot(figsize=(8, 6)); plt.gca().set_aspect("equal")
meltrate_Wang

## Compute statistics

In [ ]:
assert ds.meltrate.units == meltrate_Wilson.units == meltrate_Millan.units == meltrate_Wang.units == "m yr$^{-1}$", "wrong units"

print("Melt rate range in m/yr (own computations)")
print(f"{ds.meltrate.min():7.2f} to {ds.meltrate.max():6.2f} (our model)")
print(f"{meltrate_Wilson.min():7.2f} to {meltrate_Wilson.max():6.2f} (Wilson's data)")
print(f"{meltrate_Millan.min():7.2f} to {meltrate_Millan.max():6.2f} (Millan's data)")
print(f"{meltrate_Wang.min():7.2f} to {meltrate_Wang.max():6.2f} (Wang's data)")

print("\nAverage melt rate in m/yr")
print(f"{ds.meltrate.weighted(ds.areaC).mean():.1f} (our model)")
print(f"{meltrate_Wilson.mean():4.1f} (Wilson's data, own computation)")
print(f"{meltrate_Millan.mean():4.1f} (Millan's data, own computation)")
print(f"{meltrate_Wang.mean():4.1f} (Wang's data, own computation)")
print("10.4 ± 3.1 (Schaffer et al. 2020)")

print("\nTotal melt volume in km³/yr")
print(f"{(ds.meltrate * ds.areaC).sum() / 1e9:.1f} (our model)")
print(f"{(meltrate_Wilson * dx_Wilson**2).sum() / 1e9:4.1f} (Wilson's data, own computation)")
print(f"{(meltrate_Millan * dx_Millan**2).sum() / 1e9:4.1f} (Millan's data, own computation)")
print(f"{(meltrate_Wang * dx_Wang**2).sum() / 1e9:4.1f} (Wang's data, own computation)")
print("11.9 ± 1.6 (Wilson et al. 2017)")
print("17.8 ± 5.2 (Schaffer et al. 2020)")

## Create the figure

In [ ]:
fig, axs = plt.subplots(2, 2, sharex=True, sharey=True, constrained_layout=True, figsize=(12*cm, 14.5*cm), dpi=200)

coords = dict(x="lon", y="lat")
kwargs = dict(vmax=50, cmap=cmo.balance, add_colorbar=False)
kwargs["vmin"] = -kwargs["vmax"]
ticks = np.linspace(kwargs["vmin"], kwargs["vmax"], 5)

ax = axs[0, 0]
ds.meltrate.plot(ax=ax, **kwargs)
ice_draft = ds.glIceD * 920 / 1025
cs = ice_draft.plot.contour(ax=ax, levels=np.arange(0, ice_draft.max(), 100), cmap=cmo.ice_r, linewidths=0.75)
ax.text(-20.25, 79.38, "ice draft contours:", size="small", ha="center")
for i, c in enumerate(cs.get_edgecolors()):
    if i == 0:
        # 0 m-contour is not visible anyway
        continue
    ax.text(-20.75 + (i > 3), 79.34 - 0.04*((i-1)%3), f"{cs.levels[i]:.0f} m", color=c, size="small", ha="center")
ax.set_title("Simulation")
ax.text(-22.68, 79.83, "This study", size="small", va="top")

ax = axs[0, 1]
meltrate_Wilson.plot(ax=ax, **coords, **kwargs)
ax.set_title("Obs. 2011–2015")
ax.text(-22.68, 79.83, "Wilson et al. (2017)", size="small", va="top")

ax = axs[1, 0]
meltrate_Millan.plot(ax=ax, **coords, **kwargs)
ax.set_title("Obs. 2011–2020")
ax.text(-22.68, 79.83, "Millan et al. (2023)", size="small", va="top")

ax = axs[1, 1]
im = meltrate_Wang.plot(ax=ax, **coords, **kwargs)
ax.set_title("Obs. 2013–2022")
ax.text(-22.68, 79.83, "Wang et al. (2024)", size="small", va="top")

cbar = fig.colorbar(im, ax=axs, location="top", pad=0.03, shrink=0.8, aspect=30, extend="both", ticks=ticks)
cbar.set_label("Melt rate of the 79NG ice tongue in m yr$^{-1}$", labelpad=8, size="large")

for letter, ax in zip("abcd", axs.flat):
    title = ax.get_title()
    ds.ice_mask.plot.contour(ax=ax, levels=[0.5], colors=["0.5"], linewidths=0.75)
    ax.set_title(f"({letter}) {title}")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{-x:.0f}°W")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{round(x, 10)}°N")
    ax.set_xlim(-22.75, -19.25)
    ax.set_ylim(79.25, 79.85)
    ax.set_aspect(grille_carree)

fig.savefig("figures/fig_3_melt_rate.png", dpi=600)

The locations near the grounding line with locally low melting in the model are marked as grounded ice in BedMachine.
There, the ice topography is smoothly interpolated (see the BedMachine documentation), resulting in low melt rates.